In [ ]:
!git clone https://github.com/AMAAI-Lab/text2midi
!pip install -r text2midi/requirements.txt
!pip install --upgrade torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install music21 librosa pretty_midi laion-clap

In [2]:
import os
import pickle
import torch
import torch.nn as nn
from transformers import T5Tokenizer
from huggingface_hub import hf_hub_download
import subprocess
import sys
import scipy
import soundfile as sf
import numpy as np

[1 2 3]


In [ ]:
os.chdir('text2midi')
!touch text2midi/model/__init__.py

In [ ]:
from model.transformer_model import Transformer # Now, the import should work
import music21
from scipy.spatial.distance import cosine
import librosa;
import pretty_midi
from typing import List, Dict, Tuple
import json
import random
from datasets import load_dataset
import tempfile
import requests

# CLAP imports - you may need to install these
try:
  import laion_clap
except ImportError:
    print("Warning: laion_clap not installed. Install with: pip install laion-clap")
    laion_clap = None

In [ ]:
# Download and Load Model
repo_id = "amaai-lab/text2midi"
# Download the model.bin file
model_path = hf_hub_download(repo_id=repo_id, filename="pytorch_model.bin")
# Download the vocab_remi.pkl file
tokenizer_path = hf_hub_download(repo_id=repo_id, filename="vocab_remi.pkl")

if torch.cuda.is_available():
    device = 'cuda'
elif torch.backends.mps.is_available():
    device = 'mps'
else:
    device = 'cpu'

print(f"Using device: {device}")

# Cell 3: Initialize Tokenizer and Model
# Load the tokenizer dictionary
with open(tokenizer_path, "rb") as f:
    r_tokenizer = pickle.load(f)

# Get the vocab size
vocab_size = len(r_tokenizer)
print("Vocab size: ", vocab_size)
model = Transformer(vocab_size, 768, 8, 2048, 18, 1024, False, 8, device=device)
model.load_state_dict(torch.load(model_path, map_location=device))
model.eval()
tokenizer = T5Tokenizer.from_pretrained("google/flan-t5-base")

print('Model loaded.')